In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.1 Counting: Combinatorics and Microstate Enumeration

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.1",
    title="Counting: Combinatorics and Microstate Enumeration",
    blurb="Statistical mechanics begins with counting. From socks in a drawer to "
    "poker hands to balls in boxes, the art of enumerating configurations "
    "is exactly the art of counting a physical system's microstates — and "
    "how we count turns out to separate the classical world from the "
    "quantum.",
    difficulty="intermediate",
    estimate="140–180 min",
)

## Notebook overview

This notebook, the first of Volume V, teaches **counting** — and it means that literally.
Before any thermodynamics, before entropy or temperature, statistical mechanics rests on a
humble skill: given a system, enumerate its possible configurations. That is all a
*microstate count* is, and the entire edifice of the subject is built on top of it. Entropy
will turn out to be (the logarithm of) a number of configurations; equilibrium will turn out
to be the configuration that can happen in the most ways. So we begin by learning to count
well, patiently, from the simplest cases to a few genuinely tricky ones.

We build from the ground up. The multiplication principle — independent choices multiply —
is the seed of everything. From it grow permutations and combinations, the first fork being
whether order matters. We then push into structured counting with the worked example
everyone has an intuition for, **poker hands**, taking special care with the two places
students reliably miscount (the straight, where the ace plays both high and low, and the
need to exclude overlapping categories). And we arrive at the deepest idea in the notebook,
the one that makes counting matter for quantum physics: the count of ways to arrange objects
depends entirely on whether the objects are **distinguishable**. The very same setup —
three balls, five boxes — gives three different answers ($125$, $35$, $10$) depending only
on how we choose to count, and those three answers are precisely the three statistics that,
in Volume VII, separate classical particles from photons from electrons.

Throughout, every example carries a small **forward-pointer** to the physics it becomes, so
the statistical-mechanics context is always in view: socks drawn from a drawer are the seed
of occupation numbers, coin sequences are a two-state paramagnet, balls in boxes are
particles in energy levels. We lean on the computer in two ways the subject will use
constantly: exact counts with Python's `math.comb` and `math.perm`, and **Monte Carlo**
checks — dealing thousands of random hands with `numpy.random.default_rng` and watching the
observed frequencies converge to the counts we computed by hand.

> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the multiplication principle's products; $C(n,k)=P(n,k)/k!$; the
> sock probabilities; the poker-hand counts (exact, to the classic values); the
> stars-and-bars formula; the three statistics $125/35/10$; the birthday threshold. Many are
> cross-checked by a Monte Carlo simulation. A ✓ is strong evidence; a ✗ is a prompt to
> *locate the discrepancy*, not a verdict.
>
> **Scope.** Counting as the foundation of microstate enumeration; probability proper
> (turning counts into probabilities, expectation, variance) is [§5.2](probability.ipynb), and the large-$N$ limit
> is [§5.3](large-n-limit.ipynb). The physics — entropy, the Boltzmann distribution, the Ising model — begins at [§5.4](microstates-entropy-temperature.ipynb).
> See Schroeder, *An Introduction to Thermal Physics* (microstate counting); Graham, Knuth &
> Patashnik, *Concrete Mathematics*; and Volume VII (where the three statistics become three
> gases).

## Theory in brief

### The multiplication principle

The seed of all counting is almost too simple to state. If one choice can be made in $a$
ways and a second, independent choice in $b$ ways, then the two together can be made in

```{math}
:label: eq-multiplication
N = a \times b
```

ways, and the rule extends to any number of independent choices. Two dice show $6\times6=36$
outcomes; a three-letter string over the alphabet has $26^3$. Every count in this notebook
is, underneath, a careful application of this one principle.

### Permutations and combinations

The first crucial fork is whether **order matters**. An ordered arrangement of $k$ items
chosen from $n$ is a **permutation**, and an unordered selection is a **combination**,

```{math}
:label: eq-perm-comb
P(n,k)=\frac{n!}{(n-k)!}, \qquad C(n,k)=\binom{n}{k}=\frac{n!}{k!\,(n-k)!}=\frac{P(n,k)}{k!}.
```

The combination is the permutation with the $k!$ orderings of the chosen items divided back
out, since for an unordered selection those orderings are the same selection. (A second fork,
*with* versus *without* replacement, we meet through the examples.)

### Counting structured configurations

When a configuration must have a particular structure — a poker hand of a named type — we
count by the multiplication principle over its **independent** choices, with two recurring
cautions: do not double-count, and exclude configurations that also belong to a more
special category. This careful enumeration is the same skill, exactly, as counting the
microstates of a physical system,

```{math}
:label: eq-structured
N(\text{type}) = \prod_i (\text{independent choices})_i \ - \ (\text{overlaps with rarer types}).
```

### Distinguishable versus indistinguishable — the tenet

Here is the idea that makes counting a quantum subject. The number of ways to place $n$
objects into $k$ states (boxes) depends entirely on whether the objects are
**distinguishable**, and whether more than one may share a state,

```{math}
:label: eq-distinguishable
\underbrace{k^{\,n}}_{\text{distinguishable (MB)}}, \qquad
\underbrace{\binom{n+k-1}{k-1}}_{\substack{\text{indistinguishable,}\\ \text{any number (BE)}}}, \qquad
\underbrace{\binom{k}{n}}_{\substack{\text{indistinguishable,}\\ \text{at most one (FD)}}}.
```

The same physical question — $n$ particles among $k$ states — has **three** answers. They are
the **Maxwell–Boltzmann**, **Bose–Einstein**, and **Fermi–Dirac** counts, and the entire
distinction between classical particles, bosons (photons), and fermions (electrons) is this
single choice of how to count. We establish the counting here; Volume VII supplies the
physics that picks which rule nature uses.

## Setup

In [ ]:
from itertools import combinations, combinations_with_replacement, product
from math import comb, factorial, perm

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from ecp import combinatorics as cb
from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = cb.RED


def birthday_shared(n, days=365):
    """The probability that at least two of ``n`` people share a birthday (eq-multiplication).

    Computed as the complement of all-distinct, 1 − Π_{i=0..n-1} (days − i)/days, a pure
    application of the multiplication principle to the all-different count.

    Parameters
    ----------
    n : int
        Number of people.
    days : int, optional
        Number of equally likely birthdays (default ``365``).

    Returns
    -------
    float
        The probability of at least one shared birthday.
    """
    p_all_distinct = np.prod([(days - i) / days for i in range(n)])
    return 1.0 - p_all_distinct

## Exercise 1 — The multiplication principle and simple counts

We start as gently as possible, because everything rests here. The **multiplication
principle** {eq}`eq-multiplication` says that independent choices multiply. Roll two dice:
the first can land six ways, and for *each* of those the second can also land six ways, so
the total is $6\times6=36$ — not $6+6$ ({numref}`fig-ct-dice`). The same logic counts
three-letter strings ($26\times26\times26=26^3$) or the bytes of a computer ($2^8=256$). The
word "independent" is the whole content: the second die does not care what the first did.

**This exercise (worked).** Confirm with plain `math` arithmetic that two dice give $36$
outcomes, three dice $216$, and a three-letter string $26^3=17576$, each a product of
independent choices — and verify each count by brute-force enumeration with
`itertools.product`, letting the computer list every outcome the principle predicted.
*Forward-pointer:* a physical system of $N$ parts each with $g$ available states has $g^N$
configurations by exactly this rule — the starting point of microstate counting.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    (two_dice, three_dice, strings_3)
    == (enum_two, enum_three, enum_strings)
    == (36, 216, 17576),
    "the multiplication principle's products match brute-force enumeration (36, 216, 17576)",
)

## Exercise 2 — Permutations versus combinations

The first real distinction in counting is whether **order matters**, and it is worth slowing
down on, because it is the source of most counting mistakes. Suppose eight runners finish a
race. The number of ways to fill the three podium places — gold, silver, bronze — is a
**permutation**, $P(8,3)=8\times7\times6=336$, because here the order is the whole point:
gold-then-silver is a different outcome from silver-then-gold. But the number of ways to
choose a three-person committee from the same eight is a **combination**, $C(8,3)=56$,
because a committee is just a set — the same three people in any order are the same committee
({numref}`fig-ct-orderfork`). The two are related by {eq}`eq-perm-comb`: a combination is a
permutation with the $k!$ orderings divided back out.

**This exercise (worked).** Compute $P(8,3)$ with `math.perm` and $C(8,3)$ with `math.comb`,
and confirm exactly (no floating point) that $C(8,3)=P(8,3)/3!$. *Forward-pointer:* whether
the particles we count are ordered or not is exactly the distinguishable/indistinguishable
question we reach in Exercise 9 — the order fork *is* the classical/quantum fork.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    comb(n, k),
    perm(n, k) / factorial(k),
    "combinations are permutations divided by the k! orderings",
    rtol=0.0,
)

## Exercise 3 — Socks in a drawer: drawing without replacement

Now a count with a probability attached, and our first physical seed. A drawer holds five red
and three blue socks, and we pull two out in the dark ({numref}`fig-ct-socks`). What is the
chance both are red? The number of ways to choose any two of the eight socks is $C(8,2)=28$,
and the number of ways to choose two of the five red ones is $C(5,2)=10$, so the probability
is $10/28\approx0.357$. This is **drawing without replacement** — the second draw sees a
changed drawer — and notice that within a colour the socks are *indistinguishable*: we count
red socks, not which red sock. That is the seed of **occupation-number** counting, where we
ask how many particles occupy a state, not which ones.

**This exercise (worked).** Compute $P(\text{two red})=C(5,2)/C(8,2)$ and the probability of
a matching pair (two red *or* two blue) with `math.comb`; verify the two-red count against a
brute-force enumeration of all $C(8,2)$ pairs with `itertools.combinations`; then confirm by
a Monte Carlo simulation: draw two socks at random many times with
`numpy.random.default_rng` (the `cb.draw_without_replacement` helper) and compare the
observed frequency to the count.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    p_two_red,
    enum_two_red,
    "P(two red) from the formula matches the brute-force enumeration of all pairs",
    rtol=1e-12,
)
validate.close(
    mc_two_red,
    p_two_red,
    "the Monte Carlo sock-draw frequency matches the exact probability",
    rtol=3e-2,
)

## Exercise 4 — Coins and the binomial coefficient

Flip a coin $n$ times and ask how many of the $2^n$ possible sequences have exactly $k$
heads. The answer is the **binomial coefficient** $C(n,k)$: choosing *which* $k$ of the $n$
flips are heads is exactly choosing a $k$-subset of the $n$ positions {eq}`eq-perm-comb`.
Stacked up, the counts $C(n,0),C(n,1),\dots,C(n,n)$ form a row of **Pascal's triangle**, and
each row sums to $2^n$ because every sequence has *some* number of heads
({numref}`fig-ct-pascal`). This is our first two-state system, and it is the combinatorial
skeleton of a **paramagnet**: $n$ spins each up or down, with $C(n,k)$ arrangements having
$k$ spins up — the multiplicity we will turn into entropy at [§5.4](microstates-entropy-temperature.ipynb).

**This exercise (worked).** Confirm with `math.comb` that the number of length-$n$ sequences
with exactly $k$ heads is $C(n,k)$, that a Pascal row sums to $2^n$, and that an explicit
enumeration of all $2^n$ sequences (with `itertools.product`) reproduces the counts.
*Forward-pointer:* this two-state count is the paramagnet of [§5.4](microstates-entropy-temperature.ipynb) and the random walk of [§5.2](probability.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    pascal_row == enum_counts and sum(pascal_row) == 2**n_flips,
    "the binomial coefficients count the k-head sequences, and a Pascal row sums to 2^n",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Poker hands I: pairs and three of a kind

Now the ramp begins, with the structured-counting example everyone has a feel for. A poker
hand is five cards from a standard $52$-card deck, and the total number of hands is
$C(52,5)=2{,}598{,}960$ — order does not matter, since a hand is a set
({numref}`fig-ct-pokerhand`). The art is counting hands of a given *type* by the
multiplication principle over independent choices, and we do it slowly here because the
method is exactly microstate counting. Take **one pair**. We choose the rank of the pair
($13$ ways), choose which two of its four suits appear ($C(4,2)=6$), choose three *other*
ranks for the remaining cards ($C(12,3)=220$), and choose a suit for each of those three
($4^3=64$). Multiplying, $13\times6\times220\times64=1{,}098{,}240$.

**This exercise (worked).** Compute, with full step-by-step reasoning and `math.comb`, the
counts for one pair, two pair, and three of a kind, confirm they match the classic values,
and cross-check by dealing random hands with `numpy.random.default_rng` (the
`cb.simulate_poker` helper). *Forward-pointer:* "a rarer hand has a smaller count" is the
same statement as "a less probable macrostate has fewer microstates" — the heart of [§5.4](microstates-entropy-temperature.ipynb).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    np.array([one_pair, two_pair, three_kind]),
    np.array([1098240, 123552, 54912]),
    "the one-pair, two-pair, and three-of-a-kind counts match the classic values",
    rtol=0.0,
)
validate.close(
    tally["one pair"] / 300_000,
    one_pair / total_hands,
    "the Monte Carlo one-pair frequency matches the exact probability",
    rtol=3e-2,
)

## Exercise 6 — Poker hands II: straights and flushes, the subtle ones

These are the hands students miscount, so we go carefully. A **straight** is five cards in
consecutive rank. The subtlety is the ace: it plays *both* high and low, so the runs are
$A2345$, $23456$, …, up to $10\,J\,Q\,K\,A$ — **ten** sequences, not nine
({numref}`fig-ct-straights`). Each sequence allows any suit for each card, $4^5=1024$ ways,
giving $10\times1024=10{,}240$. But that total *includes* the straights that are also all one
suit (straight flushes), and a straight flush is a rarer, more special hand we count
separately, so we subtract the $40$ of them: $10{,}240-40=10{,}200$. A **flush** is five
cards of one suit, $C(13,5)=1287$ rank-choices per suit times $4$ suits $=5148$, again minus
the $40$ straight flushes: $5{,}108$. The two cautions — the ace wraps, and exclude the
overlap — are general microstate-counting discipline.

**This exercise (worked).** Compute the straight and flush counts with `math.comb`, showing
the ace-high-and-low enumeration and the subtraction of straight flushes explicitly, and
confirm the classic values $10{,}200$ and $5{,}108$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    np.array([straight, flush]),
    np.array([10200, 5108]),
    "the straight (10 sequences, ace high and low, minus straight flushes) and flush counts are correct",
    rtol=0.0,
)

## Exercise 7 — Poker hands III: the full ranking

We finish the ramp by counting the rarest hands and assembling the whole ranking. A **full
house** (three of one rank, two of another) is $13\times C(4,3)\times12\times C(4,2)=13
\times4\times12\times6=3{,}744$. **Four of a kind** chooses the rank ($13$) and the one
outside card ($48$ remaining), giving $624$. The **straight flush** we already have: $40$.
Laid side by side, the counts *are* the famous ranking of poker hands
({numref}`fig-ct-pokermc`), and here is the point worth pausing on: the ranking is by
**multiplicity**. A flush beats a straight, and a flush is rarer than a straight, because
there are fewer flushes — $5{,}108$ against $10{,}200$. "Rarer" and "stronger" are the same
statement, and it is a statement about counts.

**This exercise (worked).** Compute the full-house, four-of-a-kind, and straight-flush counts
with `math.comb`; assemble every category, confirm the named hands plus the high-card
remainder sum to $C(52,5)$, and plot the full probability ranking. *Forward-pointer (voice):*
a system sits in its most probable macrostate for this very reason — that macrostate is the
one realised by the most microstates. The ranking of poker hands is a small rehearsal for the
second law ([§5.3](large-n-limit.ipynb), [§5.4](microstates-entropy-temperature.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    np.array([full_house, four_kind, straight_flush]),
    np.array([3744, 624, 40]),
    "the full-house, four-of-a-kind, and straight-flush counts are correct",
    rtol=0.0,
)
validate.check(
    sum(hands.values()) == total_hands,
    "every hand is accounted for: the categories sum to C(52,5)",
)

We can also watch the counts *emerge* from random dealing. The animation below deals hands
one batch at a time and accumulates the observed frequency of each category, which settles
onto the exact probabilities (dark markers) — computation confirming the count
({numref}`fig-ct-pokermc`).

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Stars and bars: indistinguishable objects

Here is the hard, beautiful count, and the gateway to quantum statistics. How many ways can
$n$ **indistinguishable** balls be distributed among $k$ boxes, with any number allowed per
box? The trick is a change of view. Lay the $n$ balls in a row and insert $k-1$ **bars** to
divide them into $k$ groups — the first group goes in box $1$, the next in box $2$, and so on
({numref}`fig-ct-starsbars`). Every distribution corresponds to exactly one arrangement of
stars (balls) and bars, and *vice versa*, so counting distributions is the same as counting
arrangements of $n+k-1$ symbols of which $k-1$ are bars:

$$\binom{n+k-1}{k-1}.$$

Because the balls are interchangeable, only *how many* land in each box matters, not which —
precisely the situation for **identical bosons** among energy states.

**This exercise (worked).** Implement the stars-and-bars count (the `cb.stars_bars` helper),
evaluate it for several $(n,k)$, and confirm it equals $C(n+k-1,k-1)$ and an explicit
enumeration of all distributions (with `itertools.combinations_with_replacement`).
*Forward-pointer:* this is the **Bose–Einstein** count of $n$ identical particles among $k$
states (Volume VII).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array([cb.stars_bars(*e) for e in examples]),
    np.array(enum_sb),
    "the stars-and-bars formula matches explicit enumeration of every distribution",
    rtol=0.0,
)
validate.close(
    np.array(enum_sb),
    np.array([4, 21, 286]),
    "indistinguishable balls in boxes: 4, 21, 286 for (n,k)=(3,2),(5,3),(10,4)",
    rtol=0.0,
)

## Exercise 9 — The three statistics from one setup

This is the centerpiece, and the deepest idea in the notebook. Take one physical question —
place $n$ objects into $k$ states — and count it three ways, changing nothing but *what we
assume about the objects* {eq}`eq-distinguishable`. If the objects are **distinguishable**
(each carries an identity, and each independently picks a state), the count is $k^n$. If they
are **indistinguishable** with any number allowed per state, it is the stars-and-bars
$C(n+k-1,k-1)$. If they are indistinguishable *and* at most one may occupy a state, it is
$C(k,n)$. For $n=3$ objects in $k=5$ states these are $125$, $35$, and $10$ — three different
answers to one question ({numref}`fig-ct-three`).

These are the three statistics of nature. Distinguishable is **Maxwell–Boltzmann** (classical
particles); indistinguishable-any-number is **Bose–Einstein** (photons, and any integer-spin
particle); indistinguishable-at-most-one is **Fermi–Dirac** (electrons, and the exclusion
principle made into a counting rule). The entire difference between a beam of light and a
block of metal is, at this level, a choice of how to count. A reader who has counted poker
hands and stars-and-bars can already count quantum states; Volume VII supplies the physics
that says which rule applies when.

**This exercise (worked).** For $n=3$, $k=5$ compute all three counts (the `cb.count_statistics`
helper) and confirm they are $125$, $35$, $10$; then enumerate the configurations of a smaller
case ($n=2$, $k=3$) explicitly to *see* why the counts differ. *Forward-pointer:* these three
rules are the subject of Volume VII — the photon gas, the electron gas, and Bose–Einstein
condensation.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    np.array([counts["MB"], counts["BE"], counts["FD"]]),
    np.array([125, 35, 10]),
    "one setup, three counts: Maxwell–Boltzmann (125), Bose–Einstein (35), Fermi–Dirac (10)",
    rtol=0.0,
)
validate.check(
    len(mb) == 9 and len(be) == 6 and len(fd) == 3,
    "the small case n=2,k=3 enumerates to MB=9, BE=6, FD=3",
)

The enumeration below makes the difference visible. For two objects in three boxes, the
distinguishable scheme (left) keeps $AB$ and $BA$ as different — nine configurations; the
Bose scheme (centre) treats them as the same, collapsing to six; the Fermi scheme (right)
additionally forbids two in a box, leaving three ({numref}`fig-ct-three`).

In [ ]:
# (solution hidden on the public site)


## Exercise 10 — The birthday problem: when counting breaks intuition

We close with a count whose answer almost everyone gets wrong, as a reminder that careful
enumeration beats gut feeling. In a room of $n$ people, what is the chance that two share a
birthday? Intuition says you need many people — there are $365$ days, after all. But the
right count is over **pairs**, and a room of $n$ people contains $C(n,2)$ pairs, which grows
fast. It is easier to count the complement: the probability that *all* birthdays differ is
$\frac{365}{365}\cdot\frac{364}{365}\cdots\frac{365-n+1}{365}$, a multiplication-principle
product, and one minus that is the chance of a match. The surprise is that just **23** people
already cross $50\%$ ({numref}`fig-ct-birthday`).

**This exercise (student).** Evaluate the shared-birthday probability (the `birthday_shared`
helper) and confirm that $23$ people give just over $50\%$, then cross-check with a Monte
Carlo over random rooms using `numpy.random.default_rng`.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.close(
    p23, 0.507, "23 people already give >50% chance of a shared birthday", rtol=1e-2
)
validate.close(
    has_match.mean(),
    p23,
    "the Monte Carlo birthday frequency matches the exact probability",
    rtol=3e-2,
)

In [ ]:
# (solution hidden on the public site)


## Notebook summary

Counting is the foundation of statistical mechanics, and this notebook built it from the
multiplication principle up to the gateway of quantum statistics.

- **The multiplication principle** {eq}`eq-multiplication`: independent choices multiply
  ($6\times6=36$), and a system of $N$ parts with $g$ states each has $g^N$ configurations —
  microstate counting in embryo.
- **Permutations and combinations** {eq}`eq-perm-comb`: the order fork, $P(8,3)=336$ versus
  $C(8,3)=56$, with $C=P/k!$; sampling without replacement (socks, $C(5,2)/C(8,2)=0.357$); and
  the binomial coefficient counting coin sequences (the two-state paramagnet).
- **Structured counting** {eq}`eq-structured`: the full poker ranking computed exactly and
  confirmed by Monte Carlo — $1{,}098{,}240$ pairs down to $40$ straight flushes — with the
  ace-high-and-low and exclude-the-overlap subtleties. A rarer hand has a smaller count, the
  same fact that makes a system sit in its most probable macrostate.
- **Distinguishability — the tenet** {eq}`eq-distinguishable`: stars-and-bars $C(n+k-1,k-1)$,
  and the **three statistics from one setup**, $125/35/10$ for $n=3,k=5$ — Maxwell–Boltzmann,
  Bose–Einstein, Fermi–Dirac. The classical/quantum distinction is a choice of how to count.
- **A caution** (the birthday problem): $23$ people already share a birthday with probability
  $>1/2$ — count the right thing (pairs), and intuition is no substitute for enumeration.

The same multiplication principle that ranks poker hands enumerates a physical system's
microstates; the distinguishable/indistinguishable fork that separates $125$ from $35$ from
$10$ is the very thing that, in Volume VII, separates the photon gas from the electron gas.
Counting configurations is the first half of statistical mechanics. The second half is
turning those counts into **probabilities** — which is where [§5.2](probability.ipynb) begins.

## Outlook

- **Probability ([§5.2](probability.ipynb)).** From counts to probabilities: distributions, expectation, and
  variance, in the Born-rule form $\langle A\rangle$ and the uncertainty $\Delta A$ that
  quantum mechanics uses; the binomial, Poisson, and Gaussian distributions.
- **The large-$N$ limit ([§5.3](large-n-limit.ipynb)).** Stirling's approximation, the central limit theorem, and why
  the most probable configuration overwhelmingly dominates when $N\sim10^{23}$.
- **The physics begins ([§5.4](microstates-entropy-temperature.ipynb) onward).** Microstate counting becomes entropy $S=k\ln\Omega$, the
  Boltzmann distribution, and thermodynamics — emergent, not assumed.
- **The three statistics in full (Volume VII).** The photon (Bose) and electron (Fermi)
  gases, and Bose–Einstein condensation — the physics behind the three counts established here.
- **Cross-reference** Volume 0 (the computational foundations) and Volume VII (quantum
  statistics).

In [ ]:
from ecp.style import footer

footer()